# 01 — Train the caries detector (baseline and robustness arms)

Runs after `00_setup_and_sanity_check.ipynb` has passed. This is the actual
Phase 3 GPU training run: HierarchicalDet's Swin-Large + DiffusionDet
architecture, `SOLVER.MAX_ITER = 40000`, `IMS_PER_BATCH = 1` (2 OOM'd on a single T4).

**This notebook trains both arms**, selected by `TRAIN_ARM` in section 4:

| arm | `degrade_prob` | what it is |
|---|---|---|
| `baseline` | 0.0 | clean DENTEX images — the reference column |
| `robustness` | 0.7 | degradation-augmented — claim #1 of the paper |

Run both, in separate sessions, into separate `OUTPUT_DIR`s (handled
automatically). Neither arm is optional: the whole robustness claim is the
*difference* between them, so a robustness number with no baseline to compare
against says nothing.

**Read this first — two real bugs found and fixed during development, both
confirmed by actually running this pipeline (not guessed):**

1. **Do not use `hierarchialdet.dataset_mapper.DiffusionDetDatasetMapper`.**
   It unconditionally tries to open two hardcoded personal file paths from
   the original author's machine
   (`ibrahim/Diseasedataset_base_enumeration_m_t_inference_train/...`) in its
   constructor, and crashes with `FileNotFoundError` for anyone else. This
   notebook defines its own `CariesDatasetMapper` instead (below), which
   trains directly from ground-truth boxes -- confirmed the model's own
   `prepare_inferred_boxes`/`prepare_targets` gracefully fall back to
   standard (non-hierarchical-curriculum) DiffusionDet training when no
   pretrained-boxes are provided (bare `except: pass` in the source), so
   this is a correct simplification for a caries-only detector, not a hack.
2. **`cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "full_model"` (the config's
   default) is invalid in the detectron2 version this install recipe pulls.**
   Raises `ValueError: 'full_model' is not a valid GradientClipType` — this
   version only accepts `"value"` or `"norm"`. Overridden below to `"norm"`.

**Kaggle session limits**: GPU sessions have a runtime limit and a weekly GPU
quota — 40k iterations will almost certainly NOT fit in one session. This
notebook checkpoints periodically and resumes automatically; see the
"multi-session workflow" section below for exactly how to continue after a
session ends.

## 1. Setup (repeat of 00, condensed — see that notebook for the full
explanation of each step if anything here fails)

In [ ]:
# Idempotent bootstrap -- safe to re-run (session restart, or you ran the cell twice).
# The old version was `!git clone` + `%cd`: it errored on the second run, and then
# left the notebook in the wrong directory with every relative path quietly broken.
import os, subprocess, sys

REPO_URL = "https://github.com/AIscend-Research/dental-extension.git"
if not os.path.exists("src/data/degradation.py"):          # not already at the repo root
    if not os.path.isdir("dental-extension"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("dental-extension")
sys.path.insert(0, ".")

from src.utils.kaggle_env import install_deps, summarize_environment

# Installs ONLY what's missing, with numpy/torch pinned to the image's versions.
# Do NOT `pip install -r requirements-core.txt` here: it can upgrade numpy, and
# Kaggle's torch -- plus the detectron2 you are about to build against it -- is
# compiled for the numpy already in the image. The upgrade "succeeds", then torch
# dies at import with "compiled using NumPy 1.x cannot be run in NumPy 2.x".
print("installed:", install_deps() or "nothing needed -- image already has it")
for k, v in summarize_environment().items():
    print(f"  {k}: {v}")

# Build detectron2 against the image's torch. Never reinstall torch on Kaggle.
# Deliberately NOT -q: this compiles for ~10 minutes, and a silent cell that long
# is indistinguishable from a hang.
import torch
print(f"building detectron2 against torch {torch.__version__} -- expect ~10 min")

!pip install -q ninja
!pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

In [ ]:
import os
os.makedirs("models_weights", exist_ok=True)
if not os.path.exists("models_weights/swin_large_patch4_window7_224_22k.pkl"):
    # --fail: without it curl writes the error page to disk on a failed
    # download, and torch.load then dies with an unrelated-looking error
    # instead of "the download failed". Matches notebook 00.
    !curl -sL --fail "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_large_patch4_window7_224_22k.pth" \
      -o models_weights/swin_large_patch4_window7_224_22k_raw.pth
    import pickle
    RAW = "models_weights/swin_large_patch4_window7_224_22k_raw.pth"
    assert os.path.exists(RAW) and os.path.getsize(RAW) > 1e8, \
        "backbone download failed or was truncated -- is notebook internet enabled?"
    ckpt = torch.load(RAW, map_location="cpu", weights_only=False)
    assert ckpt["model"]["patch_embed.proj.weight"].shape[0] == 192, "expected Swin-Large's 192-dim embedding"
    converted = {"model": ckpt["model"], "__author__": "third_party", "matching_heuristics": True}
    with open("models_weights/swin_large_patch4_window7_224_22k.pkl", "wb") as f:
        pickle.dump(converted, f)
    os.remove(RAW)
print("backbone weights ready")

## Clone the HierarchicalDet baseline

Missing from this notebook originally -- `import_hierarchicaldet()` below needs `external/HierarchicalDet` to already exist, the same as notebook 00's step 3.


In [ ]:
!bash scripts/clone_baseline.sh

## 2. Dataset

Same as notebook 00 — set `DATA_ROOT` to wherever DENTEX actually is
(attached Kaggle Dataset, or downloaded directly).

In [ ]:
from src.utils.kaggle_env import find_dentex_root

# Finds DENTEX wherever it is mounted rather than assuming a dataset slug -- the
# path depends on what you named the Kaggle Dataset. If it is not attached, the
# error tells you how to attach it.
DATA_ROOT = str(find_dentex_root())
print("DATA_ROOT:", DATA_ROOT)

## 3. Register the dataset and define the custom mapper

`CariesDatasetMapper` reads our COCO annotations (category_id_1/2/3, the
DENTEX quadrant/enumeration/diagnosis hierarchy) into an `Instances` object
with `gt_boxes`, `gt_classes_1`, `gt_classes_2`, `gt_classes_3` — the exact
fields `DiffusionDet.forward()`'s training path reads (confirmed by reading
`hierarchialdet/detector.py`). No pretrained-boxes curriculum, no dependency
on the author's missing files.

**The mapper also implements the robustness arm.** Set `degrade_prob > 0` and
each training image is passed through `src/data/degradation.py` first — this
is what "train the detector on synthetically degraded images so it holds up"
means concretely, and it is the difference between the two arms the paper
compares. Section 4 below picks the arm.

**Why boxes go through the degradation call rather than around it**: `angle`
applies a rotation + perspective warp, so it *moves image content*. Degrading
pixels while leaving ground-truth boxes where they were would train the model
against silently misaligned labels — no crash, no warning, just a worse
model and a meaningless robustness comparison. `apply_degradations(...,
boxes=...)` returns the boxes remapped through the same homography (see
`transform_boxes()` and `tests/test_degradation.py`), which is what this
mapper uses. Degradation is applied **before** the resize, in original image
coordinates, so the existing scale factors still apply afterward.

In [ ]:
import sys, warnings, copy
warnings.filterwarnings("ignore")
from src.utils.kaggle_env import import_hierarchicaldet
add_diffusiondet_config = import_hierarchicaldet()

import cv2
import numpy as np
from detectron2.structures import Instances, Boxes
from detectron2.data import DatasetCatalog

sys.path.insert(0, ".")
from src.data.degradation import apply_degradations
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
print("registered:", {k: len(v) for k, v in split.items()})


class CariesDatasetMapper:
    """Reads DENTEX's hierarchical annotations into DiffusionDet's expected
    Instances fields. Ground-truth boxes only -- no pretrained-box curriculum
    (see the note at the top of this notebook for why that's the right call
    here, not a corner cut).

    degrade_prob controls the robustness arm: 0.0 trains on clean images (the
    baseline), >0.0 applies src/data/degradation.py to that fraction of
    training images, with ground-truth boxes remapped through the same
    geometric warp. Keeping some clean images in the mix (prob < 1.0) is
    deliberate -- the deployed model still sees good photos, and an all-
    degraded diet would trade clean-image accuracy away for nothing.
    """

    def __init__(self, target_size=800, is_train=True, degrade_prob=0.0,
                 severity_range=(0.3, 0.9), max_simultaneous=3):
        self.target_size = target_size
        self.is_train = is_train
        self.degrade_prob = degrade_prob
        self.severity_range = severity_range
        self.max_simultaneous = max_simultaneous

    def __call__(self, d):
        d = copy.deepcopy(d)
        img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
        h0, w0 = img.shape[:2]

        annos = [a for a in d.get("annotations", []) if not a.get("iscrowd", 0)]
        boxes_xywh = np.array([a["bbox"] for a in annos], dtype=np.float64).reshape(-1, 4)

        # --- robustness arm: degrade the image AND remap the boxes with it ---
        if self.is_train and self.degrade_prob > 0 and np.random.rand() < self.degrade_prob:
            res = apply_degradations(
                img,
                severity_range=self.severity_range,
                max_simultaneous=self.max_simultaneous,
                boxes=boxes_xywh,
            )
            img = res.image
            boxes_xywh = res.boxes
            # a box warped out of frame comes back zero-area -- drop it and its
            # labels together, or the class lists desynchronize from the boxes
            keep = (boxes_xywh[:, 2] > 1.0) & (boxes_xywh[:, 3] > 1.0)
            boxes_xywh = boxes_xywh[keep]
            annos = [a for a, k in zip(annos, keep) if k]

        img = cv2.resize(img, (self.target_size, self.target_size))
        scale_x, scale_y = self.target_size / w0, self.target_size / h0

        out = {
            "image": torch.as_tensor(img.transpose(2, 0, 1).astype(np.float32)),
            "height": self.target_size, "width": self.target_size,
        }
        if not self.is_train:
            return out

        inst = Instances((self.target_size, self.target_size))
        boxes, c1, c2, c3 = [], [], [], []
        for ann, (x, y, w, h) in zip(annos, boxes_xywh):
            boxes.append([x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y])
            c1.append(ann["category_id_1"]); c2.append(ann["category_id_2"]); c3.append(ann["category_id_3"])
        inst.gt_boxes = Boxes(torch.tensor(boxes, dtype=torch.float32)) if boxes else Boxes(torch.zeros(0, 4))
        inst.gt_classes_1 = torch.tensor(c1, dtype=torch.int64)
        inst.gt_classes_2 = torch.tensor(c2, dtype=torch.int64)
        inst.gt_classes_3 = torch.tensor(c3, dtype=torch.int64)
        out["instances"] = inst
        return out


# Sanity check the degraded path before spending GPU hours on it: boxes must
# move with the image, and must stay inside the frame.
_probe = CariesDatasetMapper(is_train=True, degrade_prob=1.0)(DatasetCatalog.get("custom_train_class")[0])
_pb = _probe["instances"].gt_boxes.tensor
print("degraded-sample check -- boxes:", tuple(_pb.shape),
      "| in frame:", bool(((_pb >= 0) & (_pb <= 800)).all()))

## 4. Config and trainer

`CariesTrainer` overrides `build_train_loader` to use `CariesDatasetMapper`
instead of HierarchicalDet's broken one. Everything else (optimizer,
scheduler, checkpointing, logging) is detectron2's own well-tested
`DefaultTrainer` machinery — deliberately not hand-rolled, to keep the parts
that don't need to be custom as boring/standard as possible.

In [ ]:
import logging
# detectron2 logs the ENTIRE model architecture via logging.info() every time
# CariesTrainer(cfg) is constructed -- the probe below does this twice, so that
# alone is two full model dumps before you even see a real print() statement.
# Silencing detectron2's own logger (not Python's root logger, so your own
# print()/logging calls are unaffected) stops this without hiding real errors,
# which still raise as exceptions regardless of logging level.
logging.getLogger("detectron2").setLevel(logging.WARNING)
logging.getLogger("fvcore").setLevel(logging.WARNING)

from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader

# --- WHICH ARM ARE YOU TRAINING? ---
# "baseline"   : clean images only -- the clean-data reference column
# "robustness" : degradation-augmented -- claim #1 of the paper
# Run BOTH (separate sessions, separate OUTPUT_DIRs) -- the comparison between
# them is a result, not a formality, and notebook 03 evaluates each in turn.
TRAIN_ARM = "robustness"

DEGRADE_PROB = {"baseline": 0.0, "robustness": 0.7}[TRAIN_ARM]

class CariesTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(
            cfg, mapper=CariesDatasetMapper(is_train=True, degrade_prob=DEGRADE_PROB)
        )

    def build_hooks(self):
        # detectron2's default PeriodicWriter prints metrics every 20 iterations --
        # over 40000 iterations that is 2000 print blocks, enough to bog down a
        # Kaggle notebook's output pane. Swap it for the same writers at a longer,
        # still-frequent-enough period so progress stays visible without flooding.
        from detectron2.engine import hooks as d2_hooks
        ret = [h for h in super().build_hooks() if not isinstance(h, d2_hooks.PeriodicWriter)]
        ret.append(d2_hooks.PeriodicWriter(self.build_writers(), period=100))
        # The default PeriodicCheckpointer keeps EVERY checkpoint. Each one is
        # ~3.2 GB here (282M params fp32 + AdamW optimizer state), so at
        # CHECKPOINT_PERIOD=500 that fills Kaggle's 20 GB /kaggle/working cap
        # around iteration ~3000 and the session dies with "Your notebook
        # tried to use more disk space than is available" -- confirmed on a
        # real T4 x2 run (failed after 8197s at 20.94 GB written). Resume only
        # ever needs the newest checkpoint, so cap retention. (Set the
        # attributes fvcore's PeriodicCheckpointer would set in __init__;
        # model_final.pth is never purged by the cap.)
        for h in ret:
            if isinstance(h, d2_hooks.PeriodicCheckpointer):
                h.max_to_keep = 2
                h.recent_checkpoints = []
        return ret

cfg = get_cfg()
add_diffusiondet_config(cfg)
cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
cfg.MODEL.WEIGHTS = "models_weights/swin_large_patch4_window7_224_22k.pkl"
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.DATASETS.TRAIN = ("custom_train_class",)
cfg.DATASETS.TEST = ()  # evaluation is a separate notebook (03) -- keep this one focused on training
cfg.DATALOADER.NUM_WORKERS = 0  # confirmed: NUM_WORKERS>0 crashed in development (macOS
# spawn-based multiprocessing -- DataLoader worker exited unexpectedly, likely a
# pickling issue with the mapper/cv2 in a spawned child process). Kaggle runs Linux
# (fork-based multiprocessing), which MIGHT not hit this -- if you want the speedup,
# try NUM_WORKERS=2 and watch for the same crash before trusting it; 0 is what's
# actually been verified end-to-end.

# --- confirmed-necessary override (see the note at the top) ---
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"  # "full_model" (the config default) is invalid in this detectron2 version

# --- tune these for your GPU's memory. IMS_PER_BATCH=2 OOM'd on a single T4 at 800x800;
# start at 1 with AMP enabled, only raise it if you have more headroom than that. ---
cfg.SOLVER.IMS_PER_BATCH = 1  # 2 OOM'd at 800x800 on a single T4 (~14.5GB usable) -- confirmed on real Kaggle hardware, not a guess
cfg.SOLVER.AMP.ENABLED = True  # mixed precision -- cuts memory further, usually speeds up training too
cfg.SOLVER.MAX_ITER = 40000       # per the proposal/config; will need multiple sessions, see below
cfg.SOLVER.CHECKPOINT_PERIOD = 500  # roughly every 500 iters -- tune based on the throughput you measure below
# (disk math, so raising this stays safe: only the newest 2 checkpoints are
# kept -- see build_hooks above -- so steady-state usage is ~2 x 3.2 GB plus
# the repo and backbone weights, comfortably under the 20 GB working-dir cap)

# per-arm output dir, so the two arms cannot silently overwrite each other's
# checkpoints (resume=True below would happily pick up the wrong arm's weights)
cfg.OUTPUT_DIR = f"/kaggle/working/checkpoints_{TRAIN_ARM}"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print(f"config ready, arm={TRAIN_ARM} (degrade_prob={DEGRADE_PROB}), device:", cfg.MODEL.DEVICE)
print("output dir:", cfg.OUTPUT_DIR)

## 5. Multi-session workflow — read before hitting Run

`SOLVER.MAX_ITER=40000` will not finish in one Kaggle session. Measured on
CPU during development: ~35s/iteration -> ~390 hours total (see
`docs/phase3_model_benchmarks.md`) -- GPU will be much faster but almost
certainly still multi-session. (First real GPU data point: ~2.7s/iteration
on a Kaggle T4, i.e. ~30h for the full run — measure your own session's
number with the probe below rather than trusting this one.) Two things make
this resumable:

1. **Within a session**: `trainer.resume_or_load(resume=True)` (below) checks
   `cfg.OUTPUT_DIR` for a `last_checkpoint` file and continues from there
   automatically if one exists -- confirmed locally: killing training after
   iteration 3 and rerunning with `resume=True` correctly picked back up at
   iteration 3, not 0.
2. **Across sessions** (the part that needs a manual step): `/kaggle/working/`
   is only preserved if you **Save Version** (commit) the notebook before the
   session ends. Next session: open a **new** session of this notebook, add
   the *previous version's output* as a data source (Kaggle's "Notebook
   Output Files" — the sidebar's "Add Data" supports this), copy the
   checkpoint files into **`cfg.OUTPUT_DIR`** before running the training
   cell again, so `resume_or_load(resume=True)` finds them.

   `cfg.OUTPUT_DIR` is `/kaggle/working/checkpoints_{TRAIN_ARM}` — i.e.
   `checkpoints_baseline` or `checkpoints_robustness`, **not** a plain
   `checkpoints/`. Copy into the directory matching the arm you are resuming,
   and keep the arms' directories separate. Getting this wrong fails
   silently in the worst way: `resume_or_load` finds no `last_checkpoint`,
   shrugs, and restarts from iteration 0 — you lose the session's GPU hours
   and only notice when the iteration counter reads 0. The cell below prints
   the resolved path; copy it from there rather than retyping it.

   Copy **only the newest checkpoint** (the one `last_checkpoint` names),
   not the whole directory — resume never reads the older ones, each is
   ~3.2 GB, and `/kaggle/working` has a hard 20 GB cap (a run has already
   died on exactly that, see the checkpoint-retention note in section 4):

   ```python
   import shutil, os
   src = "/kaggle/input/<previous-version-output>/checkpoints_" + TRAIN_ARM
   os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
   latest = open(os.path.join(src, "last_checkpoint")).read().strip()
   for f in [latest, "last_checkpoint"]:
       shutil.copy(os.path.join(src, f), cfg.OUTPUT_DIR)
   print("resuming from:", latest)
   ```

Benchmark your actual per-iteration time on the real GPU **before** trusting
any ETA — the ~35s/iter above is a CPU number from development, not this
notebook's hardware.

In [ ]:
import time

# Quick throughput check: time a handful of real iterations before committing to
# the full 40000. Adjust SOLVER.CHECKPOINT_PERIOD / your session budget from what
# this actually reports on YOUR GPU -- the 35s/iter in the docs is a CPU number.
PROBE_ITERS = 10

trainer = CariesTrainer(cfg)
trainer.resume_or_load(resume=True)  # picks up an existing checkpoint if present
start_iter = trainer.iter if trainer.iter else 0

# Probe relative to where we resumed from. Setting MAX_ITER to a fixed 10 meant
# that resuming from any checkpoint at iteration >= 10 ran ZERO iterations and
# then reported "0.00s/iteration -> 0.0 hours" as if it were a measurement.
cfg.SOLVER.MAX_ITER = start_iter + PROBE_ITERS
trainer = CariesTrainer(cfg)
trainer.resume_or_load(resume=True)
start_iter = trainer.iter if trainer.iter else 0

t0 = time.time()
trainer.train()
dt = time.time() - t0
done = max(0, trainer.iter + 1 - start_iter)

if done < PROBE_ITERS:
    print(f"WARNING: only {done} iteration(s) ran -- no usable timing.")
    print("Nothing was measured; do not plan a schedule from this cell.")
    print(f"(resumed at iteration {start_iter}; check cfg.OUTPUT_DIR={cfg.OUTPUT_DIR})")
else:
    per_iter = dt / done
    print(f"measured: {per_iter:.2f}s/iteration over {done} iterations on this hardware")
    print(f"extrapolated full 40000-iter run: {40000 * per_iter / 3600:.1f} hours")
    print(f"  -> at {cfg.SOLVER.CHECKPOINT_PERIOD} iters/checkpoint, "
          f"{cfg.SOLVER.CHECKPOINT_PERIOD * per_iter / 60:.1f} min between checkpoints")

## 6. The real run

Once the throughput check above looks reasonable, bump `MAX_ITER` back to
40000 (or whatever your session-budget math says to target for this
session's chunk) and rerun. Re-running this cell after a `Save Version` +
new session (with the previous checkpoint copied into `cfg.OUTPUT_DIR`) will
resume rather than restart, per the workflow above.

In [ ]:
cfg.SOLVER.MAX_ITER = 40000  # or a smaller per-session target -- your call based on the throughput above
trainer = CariesTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()
print("done at iteration:", trainer.iter)